## SEMANTICA: Vistas de KPIs
## Cada KPI queda expuesto como vista para el negocio.
## Detalle de cada indicador en docs/04_kpi.md

In [0]:
%sql
-- KPI 1: Modelos nuevos por dia (rate de innovacion)
CREATE OR REPLACE VIEW pf.semantic.vw_kpi_modelos_nuevos AS
SELECT
  fecha_id,
  COUNT(*) AS modelos_nuevos
FROM
  pf.gold.fact_metricas_diarias
WHERE
  es_primer_dia = TRUE
GROUP BY
  fecha_id
ORDER BY
  fecha_id;

In [0]:
%sql
-- KPI 2: Descargas acumuladas y su incremento por dia
CREATE OR REPLACE VIEW pf.semantic.vw_kpi_downloads_acumulados AS
SELECT
  df.fecha,
  SUM(f.downloads) AS descargas_snapshot,
  SUM(f.delta_downloads) AS incremento_del_dia
FROM
  pf.gold.fact_metricas_diarias f
    JOIN pf.gold.dim_fecha df
      ON df.fecha_id = f.fecha_id
GROUP BY
  df.fecha
ORDER BY
  df.fecha;

In [0]:
%sql
-- KPI 3: Ranking por tarea
CREATE OR REPLACE VIEW pf.semantic.vw_kpi_ranking_task AS
SELECT
  dt.pipeline_tag AS tarea,
  COUNT(DISTINCT f.model_id) AS modelos,
  SUM(f.downloads) AS descargas,
  ROUND(SUM(f.downloads) / NULLIF(COUNT(DISTINCT f.model_id), 0), 2) AS descargas_por_modelo
FROM
  pf.gold.fact_metricas_diarias f
    JOIN pf.gold.dim_task dt
      ON dt.task_id = f.task_id
GROUP BY
  dt.pipeline_tag
ORDER BY
  descargas DESC;

In [0]:
%sql
-- KPI 4: Top organizaciones por descargas
CREATE OR REPLACE VIEW pf.semantic.vw_kpi_ranking_org AS
SELECT
  do.org_id AS organizacion,
  SUM(f.downloads) AS descargas,
  COUNT(DISTINCT f.model_id) AS modelos_publicados
FROM
  pf.gold.fact_metricas_diarias f
    JOIN pf.gold.dim_organizacion do
      ON do.org_sk = f.org_sk
GROUP BY
  do.org_id
ORDER BY
  descargas DESC
LIMIT 20;

In [0]:
%sql
-- KPI 5: Distribucion por licencia
CREATE OR REPLACE VIEW pf.semantic.vw_kpi_licencias AS
SELECT
  dl.license_tag AS licencia,
  COUNT(DISTINCT f.model_id) AS modelos,
  SUM(f.downloads) AS descargas,
  ROUND(
    SUM(f.downloads)
      / NULLIF(
        (
          SELECT
            SUM(downloads)
          FROM
            pf.gold.fact_metricas_diarias
        ),
        0
      )
      * 100,
    2
  ) AS pct_descargas
FROM
  pf.gold.fact_metricas_diarias f
    JOIN pf.gold.dim_licencia dl
      ON dl.licencia_id = f.licencia_id
GROUP BY
  dl.license_tag
ORDER BY
  descargas DESC;

In [0]:
%sql
-- KPI 6: Licencias permisivas (% open-weight) sobre modelos nuevos
CREATE OR REPLACE VIEW pf.semantic.vw_kpi_licencias_permisivas AS
SELECT
  COUNT(*) AS total_modelos_nuevos,
  SUM(
    CASE
      WHEN dl.es_permissiva THEN 1
      ELSE 0
    END
  ) AS modelos_permisivos,
  ROUND(
    100
      * SUM(
        CASE
          WHEN dl.es_permissiva THEN 1
          ELSE 0
        END
      )
      / COUNT(*),
    2
  ) AS pct_permisivos
FROM
  pf.gold.fact_metricas_diarias f
    JOIN pf.gold.dim_licencia dl
      ON dl.licencia_id = f.licencia_id
WHERE
  f.es_primer_dia = TRUE;

In [0]:
%sql
-- KPI 7: Adopcion semanal (descargas promedio por modelo)
CREATE OR REPLACE VIEW pf.semantic.vw_kpi_adopcion_semanal AS
SELECT
  DATE_TRUNC('week', df.fecha) AS semana,
  ROUND(AVG(f.downloads), 2) AS descargas_promedio_por_modelo,
  COUNT(DISTINCT f.model_id) AS modelos_activos
FROM
  pf.gold.fact_metricas_diarias f
    JOIN pf.gold.dim_fecha df
      ON df.fecha_id = f.fecha_id
GROUP BY
  DATE_TRUNC('week', df.fecha)
ORDER BY
  semana;

In [0]:
%sql
-- KPI 8: Ratio likes / descargas (modelos relevantes)
CREATE OR REPLACE VIEW pf.semantic.vw_kpi_ratio_likes AS
SELECT
  f.model_id,
  MAX(f.likes) AS likes,
  MAX(f.downloads) AS downloads,
  ROUND(MAX(f.likes) / NULLIF(MAX(f.downloads), 0), 5) AS ratio_likes_descargas
FROM
  pf.gold.fact_metricas_diarias f
GROUP BY
  f.model_id
HAVING
  MAX(f.downloads) >= 10000
ORDER BY
  ratio_likes_descargas DESC
LIMIT 20;

In [0]:
%sql
-- KPI 9: Concentracion del mercado (top-10 %)
CREATE OR REPLACE VIEW pf.semantic.vw_kpi_concentracion AS
WITH latest AS (
  SELECT
    model_id,
    downloads,
    ROW_NUMBER() OVER (ORDER BY downloads DESC) AS rn
  FROM
    pf.gold.fact_metricas_diarias
  WHERE
    fecha_id
      = (
        SELECT
          MAX(fecha_id)
        FROM
          pf.gold.fact_metricas_diarias
      )
)
SELECT
  SUM(
    CASE
      WHEN rn <= 10 THEN downloads
      ELSE 0
    END
  )
    / SUM(downloads)
    * 100 AS pct_top10
FROM
  latest;

In [0]:
%sql
-- KPI 10: Calidad del pipeline (data quality)
CREATE OR REPLACE VIEW pf.semantic.vw_kpi_calidad AS
SELECT
  (
    SELECT
      COUNT(*)
    FROM
      pf.gold.fact_metricas_diarias
  ) AS filas_fact,
  (
    SELECT
      COUNT(*)
    FROM
      pf.silver.modelos
    WHERE
      has_metadata = TRUE
  ) AS silver_con_payload,
  (
    SELECT
      COUNT(*)
    FROM
      pf.gold.dim_modelo_scd2
    WHERE
      is_current = TRUE
  ) AS modelos_vigentes,
  (
    SELECT
      COUNT(*)
    FROM
      pf.gold.dim_modelo_scd2
  )
    - (
      SELECT
        COUNT(*)
      FROM
        pf.gold.dim_modelo_scd2
      WHERE
        is_current = TRUE
    ) AS versiones_historicas;

In [0]:
%sql
-- Verificacion de algunas vistas
SELECT
  *
FROM
  pf.semantic.vw_kpi_ranking_task
LIMIT 10;

In [0]:
%sql
SELECT
  *
FROM
  pf.semantic.vw_kpi_modelos_nuevos;